# Baseline Movie Recommendation System — MovieLens

This notebook implements three classic recommendation baselines on **MovieLens Latest Small**:

1. **Popularity-based** recommendations
2. **User-based collaborative filtering** (cosine similarity)
3. **Item-based collaborative filtering** (cosine similarity)

**Stack:** Pandas · Scikit-learn · Cosine Similarity

For each approach we cover methodology, generate recommendations, and evaluate concrete examples.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.3f}".format)

DATA_DIR = Path("../data/raw/ml-latest-small/ml-latest-small")
assert DATA_DIR.exists(), f"Dataset not found at {DATA_DIR.resolve()}"

RANDOM_STATE = 42
TOP_K = 10
MIN_RATING_RELEVANT = 4.0  # ratings >= this count as relevant for evaluation
N_SIMILAR = 40             # neighborhood size for CF

---
## 0. Data Loading & Train/Test Split

We hold out 20% of each user's ratings (when possible) so evaluation reflects personalization,
not only global popularity.

In [3]:
movies = pd.read_csv(DATA_DIR / "movies.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")

print(f"Movies:  {movies.shape[0]:,}")
print(f"Ratings: {ratings.shape[0]:,}")
print(f"Users:   {ratings['userId'].nunique():,}")
print(f"Rated movies: {ratings['movieId'].nunique():,}")
display(ratings.head())
display(movies.head())

Movies:  9,742
Ratings: 100,836
Users:   610
Rated movies: 9,724


,userId,movieId,rating,timestamp
0,1,1,4.000,964982703
1,1,3,4.000,964981247
2,1,6,4.000,964982224
3,1,47,5.000,964983815
4,1,50,5.000,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
def user_holdout_split(
    ratings_df: pd.DataFrame,
    test_size: float = 0.2,
    random_state: int = 42,
    min_ratings: int = 5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split ratings per user so every test user also appears in train."""
    train_parts, test_parts = [], []

    for user_id, user_ratings in ratings_df.groupby("userId"):
        if len(user_ratings) < min_ratings:
            train_parts.append(user_ratings)
            continue

        train_u, test_u = train_test_split(
            user_ratings,
            test_size=test_size,
            random_state=random_state,
        )
        train_parts.append(train_u)
        test_parts.append(test_u)

    train = pd.concat(train_parts, ignore_index=True)
    test = pd.concat(test_parts, ignore_index=True)
    return train, test


train_ratings, test_ratings = user_holdout_split(ratings, random_state=RANDOM_STATE)

print(f"Train ratings: {len(train_ratings):,}")
print(f"Test ratings:  {len(test_ratings):,}")
print(f"Train users:   {train_ratings['userId'].nunique():,}")
print(f"Test users:    {test_ratings['userId'].nunique():,}")

Train ratings: 80,419
Test ratings:  20,417
Train users:   610
Test users:    610


In [5]:
def build_user_item_matrix(ratings_df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """Pivot ratings into a dense user × movie matrix (NaN → 0 for cosine)."""
    matrix = ratings_df.pivot_table(
        index="userId",
        columns="movieId",
        values="rating",
        aggfunc="mean",
    )
    filled = matrix.fillna(0.0)
    return matrix, filled.values.astype(np.float64)


user_item, user_item_values = build_user_item_matrix(train_ratings)
user_ids = user_item.index.to_numpy()
movie_ids = user_item.columns.to_numpy()

user_id_to_pos = {uid: i for i, uid in enumerate(user_ids)}
movie_id_to_pos = {mid: i for i, mid in enumerate(movie_ids)}

print(f"User–item matrix: {user_item.shape[0]:,} users × {user_item.shape[1]:,} movies")
print(f"Density: {(user_item_values != 0).mean() * 100:.3f}%")

User–item matrix: 610 users × 8,965 movies
Density: 1.471%


In [6]:
def movie_lookup(movie_id_list: list[int] | np.ndarray) -> pd.DataFrame:
    """Attach titles/genres for a list of movieIds, preserving input order."""
    order = pd.DataFrame({"movieId": list(movie_id_list)})
    order["rank"] = np.arange(1, len(order) + 1)
    return (
        order.merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
        .sort_values("rank")
        .reset_index(drop=True)
    )


def user_seen_movies(user_id: int, ratings_df: pd.DataFrame = train_ratings) -> set[int]:
    return set(ratings_df.loc[ratings_df["userId"] == user_id, "movieId"])


def user_profile(user_id: int, n: int = 10) -> pd.DataFrame:
    """Show a user's highest-rated training movies (for qualitative context)."""
    profile = (
        train_ratings.loc[train_ratings["userId"] == user_id]
        .sort_values("rating", ascending=False)
        .head(n)
        .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
    )
    return profile[["movieId", "title", "genres", "rating"]].reset_index(drop=True)

---
## 1. Popularity-Based Recommendations

### Methodology

Recommend globally popular movies, optionally filtered to items the target user has not rated.

**Score (Bayesian-smoothed popularity):**

$$
\text{score}(i) = \frac{n_i}{n_i + m} \cdot \bar{r}_i + \frac{m}{n_i + m} \cdot \bar{r}
$$

where $n_i$ is the number of ratings for movie $i$, $\bar{r}_i$ its mean rating,
$\bar{r}$ the global mean rating, and $m$ a prior strength (here: median ratings per movie).

This dampens obscure movies with a few perfect scores while still favoring well-liked popular titles.

**Strengths:** simple, robust cold-start for new users, strong baseline.  
**Weaknesses:** no personalization; reinforces the head of the long tail.

In [7]:
def build_popularity_table(ratings_df: pd.DataFrame) -> pd.DataFrame:
    global_mean = ratings_df["rating"].mean()
    stats = (
        ratings_df.groupby("movieId", as_index=False)
        .agg(n_ratings=("rating", "size"), avg_rating=("rating", "mean"))
    )
    m = float(stats["n_ratings"].median())
    stats["score"] = (
        (stats["n_ratings"] / (stats["n_ratings"] + m)) * stats["avg_rating"]
        + (m / (stats["n_ratings"] + m)) * global_mean
    )
    stats["prior_m"] = m
    stats["global_mean"] = global_mean
    return stats.sort_values("score", ascending=False).reset_index(drop=True)


popularity = build_popularity_table(train_ratings)
popularity_ranked = popularity.merge(
    movies[["movieId", "title", "genres"]], on="movieId", how="left"
)

print(f"Bayesian prior m (median ratings/movie): {popularity['prior_m'].iloc[0]:.1f}")
print(f"Global mean rating: {popularity['global_mean'].iloc[0]:.3f}")
print("\nTop 15 globally popular movies (train):")
display(
    popularity_ranked[["movieId", "title", "genres", "n_ratings", "avg_rating", "score"]]
    .head(15)
)

Bayesian prior m (median ratings/movie): 2.0
Global mean rating: 3.503

Top 15 globally popular movies (train):


,movieId,title,genres,n_ratings,avg_rating,score
0,177593,"Three Billboards Outside Ebbing, Missouri (2017)",Crime|Drama,8,4.750,4.501
1,1178,Paths of Glory (1957),Drama|War,9,4.667,4.455
2,92535,Louis C.K.: Live at the Beacon Theater (2011),Comedy,8,4.688,4.451
3,3451,Guess Who's Coming to Dinner (1967),Drama,8,4.688,4.451
4,6460,"Trial, The (Procès, Le) (1962)",Drama,4,4.875,4.418
5,318,"Shawshank Redemption, The (1994)",Crime|Drama,249,4.416,4.408
6,4334,Yi Yi (2000),Drama,3,5.000,4.401
7,3201,Five Easy Pieces (1970),Drama,7,4.643,4.390
8,2239,Swept Away (Travolti da un insolito destino nell'azzurro mare d'Agosto) (1975),Comedy|Drama,6,4.667,4.376
9,1217,Ran (1985),Drama|War,12,4.500,4.358


In [8]:
def recommend_popularity(
    user_id: int,
    k: int = TOP_K,
    popularity_df: pd.DataFrame = popularity,
) -> pd.DataFrame:
    """Return top-k popular movies the user has not rated in train."""
    seen = user_seen_movies(user_id)
    candidates = popularity_df.loc[~popularity_df["movieId"].isin(seen)].head(k)
    out = movie_lookup(candidates["movieId"].to_numpy())
    out["score"] = candidates["score"].to_numpy()
    out["n_ratings"] = candidates["n_ratings"].to_numpy()
    out["avg_rating"] = candidates["avg_rating"].to_numpy()
    return out


# Example user
EXAMPLE_USER = int(
    train_ratings.groupby("userId").size().sort_values(ascending=False).index[10]
)
print(f"Example userId: {EXAMPLE_USER}")
print("User taste profile (top-rated in train):")
display(user_profile(EXAMPLE_USER, n=10))

print(f"\nPopularity recommendations for user {EXAMPLE_USER}:")
pop_recs = recommend_popularity(EXAMPLE_USER, k=TOP_K)
display(pop_recs)

Example userId: 249
User taste profile (top-rated in train):


,movieId,title,genres,rating
0,88129,Drive (2011),Crime|Drama|Film-Noir|Thriller,5.000
1,139385,The Revenant (2015),Adventure|Drama,5.000
2,96821,"Perks of Being a Wallflower, The (2012)",Drama|Romance,5.000
3,1201,"Good, the Bad and the Ugly, The (Buono, il brutto, il cattivo, Il) (1966)",Action|Adventure|Western,5.000
4,7153,"Lord of the Rings: The Return of the King, The (2003)",Action|Adventure|Drama|Fantasy,5.000
5,122882,Mad Max: Fury Road (2015),Action|Adventure|Sci-Fi|Thriller,5.000
6,94959,Moonrise Kingdom (2012),Comedy|Drama|Romance,5.000
7,164179,Arrival (2016),Sci-Fi,5.000
8,93838,The Raid: Redemption (2011),Action|Crime,5.000
9,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,5.000



Popularity recommendations for user 249:


,movieId,rank,title,genres,score,n_ratings,avg_rating
0,177593,1,"Three Billboards Outside Ebbing, Missouri (2017)",Crime|Drama,4.501,8,4.750
1,92535,2,Louis C.K.: Live at the Beacon Theater (2011),Comedy,4.451,8,4.688
2,3451,3,Guess Who's Coming to Dinner (1967),Drama,4.451,8,4.688
3,6460,4,"Trial, The (Procès, Le) (1962)",Drama,4.418,4,4.875
4,318,5,"Shawshank Redemption, The (1994)",Crime|Drama,4.408,249,4.416
5,4334,6,Yi Yi (2000),Drama,4.401,3,5.000
6,3201,7,Five Easy Pieces (1970),Drama,4.390,7,4.643
7,2239,8,Swept Away (Travolti da un insolito destino nell'azzurro mare d'Agosto) (1975),Comedy|Drama,4.376,6,4.667
8,1217,9,Ran (1985),Drama|War,4.358,12,4.500
9,306,10,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,4.344,14,4.464


---
## 2. User-Based Collaborative Filtering

### Methodology

1. Represent each user as a vector of ratings over movies (unrated → 0).
2. Compute **cosine similarity** between the target user and all other users
   (`sklearn.metrics.pairwise.cosine_similarity`).
3. Select the $N$ nearest neighbors with positive similarity.
4. Score unseen movies by a similarity-weighted average of neighbor ratings:

$$
\hat{r}_{u,i} = \frac{\sum_{v \in N(u)} \text{sim}(u,v)\, r_{v,i}}{\sum_{v \in N(u)} |\text{sim}(u,v)|}
$$

5. Rank unseen movies by $\hat{r}_{u,i}$ and return top-$k$.

**Strengths:** personalizes from peer taste.  
**Weaknesses:** sparse users have weak neighborhoods; similarity over raw ratings can be noisy.

In [9]:
# Precompute full user–user cosine similarity on the training matrix
user_sim = cosine_similarity(user_item_values)
np.fill_diagonal(user_sim, 0.0)

print(f"User similarity matrix: {user_sim.shape}")
print(
    f"Mean off-diagonal similarity: "
    f"{user_sim[np.triu_indices_from(user_sim, k=1)].mean():.4f}"
)

User similarity matrix: (610, 610)
Mean off-diagonal similarity: 0.0811


In [10]:
def recommend_user_based(
    user_id: int,
    k: int = TOP_K,
    n_neighbors: int = N_SIMILAR,
) -> pd.DataFrame:
    """User-based CF recommendations via cosine neighborhood."""
    if user_id not in user_id_to_pos:
        raise ValueError(f"userId {user_id} not in training set")

    u_pos = user_id_to_pos[user_id]
    sims = user_sim[u_pos]

    # Top neighbors with positive similarity
    neighbor_idx = np.argsort(sims)[::-1][:n_neighbors]
    neighbor_sims = sims[neighbor_idx]
    mask = neighbor_sims > 0
    neighbor_idx = neighbor_idx[mask]
    neighbor_sims = neighbor_sims[mask]

    if len(neighbor_idx) == 0:
        return pd.DataFrame(columns=["rank", "movieId", "title", "genres", "score"])

    neighbor_ratings = user_item_values[neighbor_idx]  # (n_neighbors, n_movies)
    weighted = neighbor_sims @ neighbor_ratings
    denom = np.abs(neighbor_sims) @ (neighbor_ratings != 0).astype(np.float64)
    scores = np.divide(weighted, denom, out=np.zeros_like(weighted), where=denom > 0)

    seen = user_seen_movies(user_id)
    seen_pos = [movie_id_to_pos[m] for m in seen if m in movie_id_to_pos]
    scores[seen_pos] = -np.inf

    top_pos = np.argsort(scores)[::-1][:k]
    top_pos = top_pos[np.isfinite(scores[top_pos])]

    out = movie_lookup(movie_ids[top_pos])
    out["score"] = scores[top_pos]
    out["n_neighbors_used"] = len(neighbor_idx)
    return out


print(f"User-based CF recommendations for user {EXAMPLE_USER}:")
user_cf_recs = recommend_user_based(EXAMPLE_USER, k=TOP_K)
display(user_cf_recs)

User-based CF recommendations for user 249:


,movieId,rank,title,genres,score,n_neighbors_used
0,26401,1,Last Hurrah for Chivalry (Hao xia) (1979),Action|Drama,5.000,40
1,184245,2,De platte jungle (1978),Documentary,5.000,40
2,2318,3,Happiness (1998),Comedy|Drama,5.000,40
3,4794,4,Opera (1987),Crime|Horror|Mystery,5.000,40
4,64499,5,Che: Part One (2008),Drama|War,5.000,40
5,140265,6,George Carlin: Jammin' in New York (1992),Comedy,5.000,40
6,141816,7,12 Chairs (1976),Adventure|Comedy,5.000,40
7,26169,8,Branded to Kill (Koroshi no rakuin) (1967),Action|Crime|Drama,5.000,40
8,138966,9,Nasu: Summer in Andalusia (2003),Animation,5.000,40
9,138835,10,Return to Treasure Island (1988),Adventure|Animation|Comedy,5.000,40


In [11]:
def show_nearest_users(user_id: int, n: int = 5) -> pd.DataFrame:
    """Inspect nearest neighbors for qualitative evaluation."""
    u_pos = user_id_to_pos[user_id]
    sims = user_sim[u_pos]
    top = np.argsort(sims)[::-1][:n]
    rows = []
    for pos in top:
        nb = int(user_ids[pos])
        n_ratings = int((user_item_values[pos] > 0).sum())
        rows.append({
            "neighbor_userId": nb,
            "cosine_similarity": float(sims[pos]),
            "n_train_ratings": n_ratings,
            "mean_rating": float(user_item_values[pos][user_item_values[pos] > 0].mean()),
        })
    return pd.DataFrame(rows)


print(f"Nearest neighbors of user {EXAMPLE_USER}:")
display(show_nearest_users(EXAMPLE_USER, n=5))

Nearest neighbors of user 249:


,neighbor_userId,cosine_similarity,n_train_ratings,mean_rating
0,68,0.397,1008,3.209
1,62,0.394,292,4.099
2,380,0.392,974,3.692
3,298,0.391,751,2.397
4,610,0.390,1041,3.695


---
## 3. Item-Based Collaborative Filtering

### Methodology

1. Represent each movie as a vector of ratings across users (transpose of the user–item matrix).
2. Compute **item–item cosine similarity**.
3. For a target user, score each unseen movie $i$ from the user's rated items $j$:

$$
\hat{r}_{u,i} = \frac{\sum_{j \in I(u)} \text{sim}(i,j)\, r_{u,j}}{\sum_{j \in I(u)} |\text{sim}(i,j)|}
$$

using only the $N$ most similar rated items for efficiency/stability.

4. Rank by predicted score and return top-$k$ unseen movies.

**Strengths:** item similarities are often more stable than user neighborhoods; explanations are intuitive ("because you liked X").  
**Weaknesses:** expensive item–item matrix; cold-start for brand-new movies.

In [12]:
# Item vectors = columns of the user–item matrix → transpose for cosine_similarity
item_sim = cosine_similarity(user_item_values.T)
np.fill_diagonal(item_sim, 0.0)

print(f"Item similarity matrix: {item_sim.shape}")
print(
    f"Mean off-diagonal similarity: "
    f"{item_sim[np.triu_indices_from(item_sim, k=1)].mean():.4f}"
)

Item similarity matrix: (8965, 8965)
Mean off-diagonal similarity: 0.0498


In [13]:
def recommend_item_based(
    user_id: int,
    k: int = TOP_K,
    n_similar: int = N_SIMILAR,
) -> pd.DataFrame:
    """Item-based CF recommendations via cosine neighborhood."""
    if user_id not in user_id_to_pos:
        raise ValueError(f"userId {user_id} not in training set")

    u_pos = user_id_to_pos[user_id]
    user_ratings_vec = user_item_values[u_pos]
    rated_pos = np.where(user_ratings_vec > 0)[0]

    if len(rated_pos) == 0:
        return pd.DataFrame(columns=["rank", "movieId", "title", "genres", "score"])

    scores = np.zeros(len(movie_ids), dtype=np.float64)
    weights = np.zeros(len(movie_ids), dtype=np.float64)

    for j in rated_pos:
        sims = item_sim[j]
        # Restrict to top similar items for this rated movie
        top_idx = np.argpartition(sims, -n_similar)[-n_similar:]
        top_idx = top_idx[sims[top_idx] > 0]
        if len(top_idx) == 0:
            continue
        r_uj = user_ratings_vec[j]
        scores[top_idx] += sims[top_idx] * r_uj
        weights[top_idx] += np.abs(sims[top_idx])

    preds = np.divide(scores, weights, out=np.zeros_like(scores), where=weights > 0)
    preds[rated_pos] = -np.inf

    top_pos = np.argsort(preds)[::-1][:k]
    top_pos = top_pos[np.isfinite(preds[top_pos]) & (preds[top_pos] > 0)]

    out = movie_lookup(movie_ids[top_pos])
    out["score"] = preds[top_pos]
    return out


print(f"Item-based CF recommendations for user {EXAMPLE_USER}:")
item_cf_recs = recommend_item_based(EXAMPLE_USER, k=TOP_K)
display(item_cf_recs)

Item-based CF recommendations for user 249:


,movieId,rank,title,genres,score
0,1207,1,To Kill a Mockingbird (1962),Drama,5.000
1,2788,2,Monty Python's And Now for Something Completely Different (1971),Comedy,5.000
2,5291,3,Rashomon (Rashômon) (1950),Crime|Drama|Mystery,5.000
3,89087,4,Colombiana (2011),Action|Adventure|Drama|Thriller,5.000
4,2712,5,Eyes Wide Shut (1999),Drama|Mystery|Thriller,5.000
5,8937,6,Friday Night Lights (2004),Action|Drama,5.000
6,112290,7,Boyhood (2014),Drama,5.000
7,105844,8,12 Years a Slave (2013),Drama,5.000
8,8228,9,"Maltese Falcon, The (a.k.a. Dangerous Female) (1931)",Mystery,5.000
9,92535,10,Louis C.K.: Live at the Beacon Theater (2011),Comedy,5.000


In [14]:
def similar_movies(movie_id: int, n: int = 10) -> pd.DataFrame:
    """Show nearest movies for qualitative item-based evaluation."""
    if movie_id not in movie_id_to_pos:
        raise ValueError(f"movieId {movie_id} not in training matrix")
    pos = movie_id_to_pos[movie_id]
    sims = item_sim[pos]
    top = np.argsort(sims)[::-1][:n]
    out = movie_lookup(movie_ids[top])
    out["cosine_similarity"] = sims[top]
    return out


# Anchor on the example user's favorite training movie
fav = user_profile(EXAMPLE_USER, n=1).iloc[0]
print(f"Movies similar to: {fav['title']} (movieId={int(fav['movieId'])})")
display(similar_movies(int(fav["movieId"]), n=10))

Movies similar to: Drive (2011) (movieId=88129)


,movieId,rank,title,genres,cosine_similarity
0,93840,1,"Cabin in the Woods, The (2012)",Comedy|Horror|Sci-Fi|Thriller,0.529
1,6709,2,Once Upon a Time in Mexico (2003),Action|Adventure|Crime|Thriller,0.523
2,95875,3,Total Recall (2012),Action|Sci-Fi|Thriller,0.507
3,97306,4,Seven Psychopaths (2012),Comedy|Crime,0.501
4,111364,5,Godzilla (2014),Action|Adventure|Sci-Fi|IMAX,0.501
5,92420,6,Chronicle (2012),Action|Sci-Fi|Thriller,0.483
6,96811,7,End of Watch (2012),Crime|Drama|Thriller,0.480
7,57669,8,In Bruges (2008),Comedy|Crime|Drama|Thriller,0.480
8,88140,9,Captain America: The First Avenger (2011),Action|Adventure|Sci-Fi|Thriller|War,0.478
9,72011,10,Up in the Air (2009),Drama|Romance,0.475


---
## 4. Side-by-Side Recommendations (Same User)

Compare the three strategies for the same example user.

In [15]:
def side_by_side(user_id: int, k: int = TOP_K) -> pd.DataFrame:
    pop = recommend_popularity(user_id, k=k)[["title"]].rename(columns={"title": "popularity"})
    ucf = recommend_user_based(user_id, k=k)[["title"]].rename(columns={"title": "user_cf"})
    icf = recommend_item_based(user_id, k=k)[["title"]].rename(columns={"title": "item_cf"})

    # Align lengths
    n = max(len(pop), len(ucf), len(icf), k)
    def pad(s: pd.Series) -> pd.Series:
        return s.reindex(range(n)).fillna("—")

    return pd.DataFrame({
        "rank": np.arange(1, n + 1),
        "popularity": pad(pop["popularity"]),
        "user_cf": pad(ucf["user_cf"]),
        "item_cf": pad(icf["item_cf"]),
    })


print(f"User {EXAMPLE_USER} — profile reminder:")
display(user_profile(EXAMPLE_USER, n=8))
print("\nTop recommendations by method:")
display(side_by_side(EXAMPLE_USER, k=TOP_K))

User 249 — profile reminder:


,movieId,title,genres,rating
0,88129,Drive (2011),Crime|Drama|Film-Noir|Thriller,5.000
1,139385,The Revenant (2015),Adventure|Drama,5.000
2,96821,"Perks of Being a Wallflower, The (2012)",Drama|Romance,5.000
3,1201,"Good, the Bad and the Ugly, The (Buono, il brutto, il cattivo, Il) (1966)",Action|Adventure|Western,5.000
4,7153,"Lord of the Rings: The Return of the King, The (2003)",Action|Adventure|Drama|Fantasy,5.000
5,122882,Mad Max: Fury Road (2015),Action|Adventure|Sci-Fi|Thriller,5.000
6,94959,Moonrise Kingdom (2012),Comedy|Drama|Romance,5.000
7,164179,Arrival (2016),Sci-Fi,5.000



Top recommendations by method:


,rank,popularity,user_cf,item_cf
0,1,"Three Billboards Outside Ebbing, Missouri (2017)",Last Hurrah for Chivalry (Hao xia) (1979),To Kill a Mockingbird (1962)
1,2,Louis C.K.: Live at the Beacon Theater (2011),De platte jungle (1978),Monty Python's And Now for Something Completely Different (1971)
2,3,Guess Who's Coming to Dinner (1967),Happiness (1998),Rashomon (Rashômon) (1950)
3,4,"Trial, The (Procès, Le) (1962)",Opera (1987),Colombiana (2011)
4,5,"Shawshank Redemption, The (1994)",Che: Part One (2008),Eyes Wide Shut (1999)
5,6,Yi Yi (2000),George Carlin: Jammin' in New York (1992),Friday Night Lights (2004)
6,7,Five Easy Pieces (1970),12 Chairs (1976),Boyhood (2014)
7,8,Swept Away (Travolti da un insolito destino nell'azzurro mare d'Agosto) (1975),Branded to Kill (Koroshi no rakuin) (1967),12 Years a Slave (2013)
8,9,Ran (1985),Nasu: Summer in Andalusia (2003),"Maltese Falcon, The (a.k.a. Dangerous Female) (1931)"
9,10,Three Colors: Red (Trois couleurs: Rouge) (1994),Return to Treasure Island (1988),Louis C.K.: Live at the Beacon Theater (2011)


---
## 5. Evaluation on Held-Out Ratings

### Protocol

For each test user:

1. Treat test movies with rating $\geq 4.0$ as **relevant**.
2. Generate top-$K$ recommendations (excluding train history).
3. Compute:
   - **Precision@K** = $|\text{hits}| / K$
   - **Recall@K** = $|\text{hits}| / |\text{relevant}|$
   - **Hit Rate@K** = $1$ if at least one relevant item is recommended

We evaluate on a sample of users for speed while still producing comparable averages.

In [16]:
def evaluate_recommender(
    recommend_fn,
    test_df: pd.DataFrame,
    k: int = TOP_K,
    min_rating: float = MIN_RATING_RELEVANT,
    max_users: int = 100,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Compute Precision@K, Recall@K, HitRate@K over sampled test users."""
    rng = np.random.default_rng(random_state)

    # Users with at least one relevant held-out item
    relevant = test_df.loc[test_df["rating"] >= min_rating]
    candidate_users = relevant["userId"].unique()
    candidate_users = [u for u in candidate_users if u in user_id_to_pos]

    if len(candidate_users) > max_users:
        candidate_users = rng.choice(candidate_users, size=max_users, replace=False)

    rows = []
    for user_id in candidate_users:
        truth = set(
            relevant.loc[relevant["userId"] == user_id, "movieId"].tolist()
        )
        if not truth:
            continue

        try:
            recs = recommend_fn(int(user_id), k=k)
        except Exception:
            continue

        if recs.empty:
            pred = set()
        else:
            pred = set(recs["movieId"].tolist())

        hits = pred & truth
        rows.append({
            "userId": int(user_id),
            "n_relevant": len(truth),
            "n_hits": len(hits),
            "precision_at_k": len(hits) / k,
            "recall_at_k": len(hits) / len(truth),
            "hit_rate_at_k": float(len(hits) > 0),
        })

    return pd.DataFrame(rows)


def summarize_eval(per_user: pd.DataFrame, method: str) -> dict:
    return {
        "method": method,
        "n_users": len(per_user),
        "precision@k": per_user["precision_at_k"].mean() if len(per_user) else np.nan,
        "recall@k": per_user["recall_at_k"].mean() if len(per_user) else np.nan,
        "hit_rate@k": per_user["hit_rate_at_k"].mean() if len(per_user) else np.nan,
    }

In [17]:
EVAL_USERS = 80  # sample size for tractable runtime

print(f"Evaluating on up to {EVAL_USERS} test users | K={TOP_K} | relevant ≥ {MIN_RATING_RELEVANT}")

eval_pop = evaluate_recommender(recommend_popularity, test_ratings, max_users=EVAL_USERS)
eval_ucf = evaluate_recommender(recommend_user_based, test_ratings, max_users=EVAL_USERS)
eval_icf = evaluate_recommender(recommend_item_based, test_ratings, max_users=EVAL_USERS)

results = pd.DataFrame([
    summarize_eval(eval_pop, "popularity"),
    summarize_eval(eval_ucf, "user_based_cf"),
    summarize_eval(eval_icf, "item_based_cf"),
])

display(results)

Evaluating on up to 80 test users | K=10 | relevant ≥ 4.0


,method,n_users,precision@k,recall@k,hit_rate@k
0,popularity,80,0.010,0.005,0.087
1,user_based_cf,80,0.001,0.002,0.013
2,item_based_cf,80,0.013,0.013,0.100


In [18]:
# Qualitative hit inspection for the example user (if present in test)
def explain_hits(user_id: int, recommend_fn, method_name: str, k: int = TOP_K) -> None:
    truth = test_ratings.loc[
        (test_ratings["userId"] == user_id) & (test_ratings["rating"] >= MIN_RATING_RELEVANT),
        ["movieId", "rating"],
    ].merge(movies[["movieId", "title"]], on="movieId", how="left")

    recs = recommend_fn(user_id, k=k)
    hits = recs.merge(truth, on="movieId", how="inner", suffixes=("_rec", "_test"))

    print(f"\n=== {method_name} | user {user_id} ===")
    print(f"Relevant held-out movies: {len(truth)}")
    display(truth.sort_values("rating", ascending=False).head(10))
    print(f"Hits in top-{k}: {len(hits)}")
    display(hits[["rank", "title_rec", "score", "rating"]].rename(columns={"title_rec": "title"}) if len(hits) else hits)


if EXAMPLE_USER in set(test_ratings["userId"]):
    explain_hits(EXAMPLE_USER, recommend_popularity, "Popularity")
    explain_hits(EXAMPLE_USER, recommend_user_based, "User-based CF")
    explain_hits(EXAMPLE_USER, recommend_item_based, "Item-based CF")
else:
    print(f"Example user {EXAMPLE_USER} has no held-out ratings; skip qualitative hit demo.")


=== Popularity | user 249 ===
Relevant held-out movies: 95


,movieId,rating,title
2,106920,5.000,Her (2013)
9,34405,5.000,Serenity (2005)
19,1196,5.000,Star Wars: Episode V - The Empire Strikes Back (1980)
28,51662,5.000,300 (2007)
51,166528,5.000,Rogue One: A Star Wars Story (2016)
40,112556,5.000,Gone Girl (2014)
45,293,5.000,Léon: The Professional (a.k.a. The Professional) (Léon) (1994)
15,122922,4.500,Doctor Strange (2016)
18,858,4.500,"Godfather, The (1972)"
25,59315,4.500,Iron Man (2008)


Hits in top-10: 1


,rank,title,score,rating
0,5,"Shawshank Redemption, The (1994)",4.408,4.500



=== User-based CF | user 249 ===
Relevant held-out movies: 95


...

,movieId,rank,title_rec,genres,score,n_neighbors_used,rating,title_test



=== Item-based CF | user 249 ===
Relevant held-out movies: 95


,movieId,rating,title
2,106920,5.000,Her (2013)
9,34405,5.000,Serenity (2005)
19,1196,5.000,Star Wars: Episode V - The Empire Strikes Back (1980)
28,51662,5.000,300 (2007)
51,166528,5.000,Rogue One: A Star Wars Story (2016)
40,112556,5.000,Gone Girl (2014)
45,293,5.000,Léon: The Professional (a.k.a. The Professional) (Léon) (1994)
15,122922,4.500,Doctor Strange (2016)
18,858,4.500,"Godfather, The (1972)"
25,59315,4.500,Iron Man (2008)


Hits in top-10: 0


,movieId,rank,title_rec,genres,score,rating,title_test


---
## 6. Additional Example Users

Run the same qualitative comparison on a few more users with different activity levels.

In [19]:
activity = train_ratings.groupby("userId").size().rename("n_ratings").reset_index()
activity = activity.sort_values("n_ratings")

# Low / median / high activity users present in both train and test
test_user_set = set(test_ratings["userId"])
activity = activity.loc[activity["userId"].isin(test_user_set)].reset_index(drop=True)

example_users = {
    "low_activity": int(activity.iloc[len(activity) // 10]["userId"]),
    "median_activity": int(activity.iloc[len(activity) // 2]["userId"]),
    "high_activity": int(activity.iloc[int(len(activity) * 0.9)]["userId"]),
}

for label, uid in example_users.items():
    n = int(activity.loc[activity["userId"] == uid, "n_ratings"].iloc[0])
    print(f"\n{'=' * 70}")
    print(f"{label.upper()} — userId={uid} | train ratings={n}")
    print("Profile:")
    display(user_profile(uid, n=5))
    print("Recommendations:")
    display(side_by_side(uid, k=5))


LOW_ACTIVITY — userId=180 | train ratings=19
Profile:


,movieId,title,genres,rating
0,1213,Goodfellas (1990),Crime|Drama,4.500
1,2959,Fight Club (1999),Action|Crime|Drama|Thriller,4.500
2,4993,"Lord of the Rings: The Fellowship of the Ring, The (2001)",Adventure|Fantasy,4.500
3,1198,Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981),Action|Adventure,4.000
4,1197,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.000


Recommendations:


,rank,popularity,user_cf,item_cf
0,1,"Three Billboards Outside Ebbing, Missouri (2017)",True Romance (1993),One Flew Over the Cuckoo's Nest (1975)
1,2,Paths of Glory (1957),Real Genius (1985),"Shining, The (1980)"
2,3,Louis C.K.: Live at the Beacon Theater (2011),"Postman, The (Postino, Il) (1994)",Inglourious Basterds (2009)
3,4,Guess Who's Coming to Dinner (1967),Wild Tales (2014),"Graduate, The (1967)"
4,5,"Trial, The (Procès, Le) (1962)",Spy Kids (2001),Casino (1995)



MEDIAN_ACTIVITY — userId=143 | train ratings=56
Profile:


...

,rank,popularity,user_cf,item_cf
0,1,"Three Billboards Outside Ebbing, Missouri (2017)",Babes in Toyland (1934),Dead Men Don't Wear Plaid (1982)
1,2,Paths of Glory (1957),"Red Violin, The (Violon rouge, Le) (1998)","Affair of the Necklace, The (2001)"
2,3,Louis C.K.: Live at the Beacon Theater (2011),"Duchess, The (2008)",Notorious C.H.O. (2002)
3,4,Guess Who's Coming to Dinner (1967),Mystery Men (1999),Heaven Can Wait (1978)
4,5,"Trial, The (Procès, Le) (1962)",Stop Making Sense (1984),Turner & Hooch (1989)



HIGH_ACTIVITY — userId=202 | train ratings=322
Profile:


,movieId,title,genres,rating
0,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery,5.000
1,2948,From Russia with Love (1963),Action|Adventure|Thriller,5.000
2,2959,Fight Club (1999),Action|Crime|Drama|Thriller,5.000
3,2352,"Big Chill, The (1983)",Comedy|Drama,5.000
4,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,5.000


Recommendations:


,rank,popularity,user_cf,item_cf
0,1,"Three Billboards Outside Ebbing, Missouri (2017)",Guys and Dolls (1955),"Great Yokai War, The (Yôkai daisensô) (2005)"
1,2,Paths of Glory (1957),Crossing Delancey (1988),Stander (2003)
2,3,Louis C.K.: Live at the Beacon Theater (2011),"Impostors, The (1998)",Knights of Badassdom (2013)
3,4,Guess Who's Coming to Dinner (1967),Safety Last! (1923),Zodiac (2007)
4,5,"Trial, The (Procès, Le) (1962)",Adam's Rib (1949),Red Dawn (2012)


---
## 7. Summary & Takeaways

| Approach | Idea | Best for | Main limitation |
|---|---|---|---|
| **Popularity** | Recommend globally liked movies | Cold-start users, strong baseline | No personalization |
| **User-based CF** | Find similar users, borrow their tastes | Users with clear peer neighborhoods | Sparse / atypical users |
| **Item-based CF** | Find movies similar to ones you liked | Stable personalization, explainability | New / rarely rated items |

### Practical notes

1. **Popularity is hard to beat** on Precision@K for MovieLens because popular titles dominate held-out positives.
2. **CF adds personalization** — inspect side-by-side lists: user/item CF should diverge from the global chart when the user has niche tastes.
3. **Cosine on raw ratings** treats missing values as zeros; mean-centering or implicit feedback (binarized likes) are common next upgrades.
4. **Next steps for this repo:** matrix factorization (SVD), content/genre features, and graph-based retrieval for Graph RAG.

In [20]:
print("Baseline recommendation notebook complete.")
print(f"Example user evaluated qualitatively: {EXAMPLE_USER}")
print("Aggregate metrics:")
display(results)

Baseline recommendation notebook complete.
Example user evaluated qualitatively: 249
Aggregate metrics:


,method,n_users,precision@k,recall@k,hit_rate@k
0,popularity,80,0.010,0.005,0.087
1,user_based_cf,80,0.001,0.002,0.013
2,item_based_cf,80,0.013,0.013,0.100
